# 05 · Corridors: route check R1 and Gate C1

Fork `fork/corridors-dose`, Part 1. Every rule is in
[`docs/successors/corridors-prespec.md`](../../docs/successors/corridors-prespec.md); outcomes are
recorded there (§8). **This notebook loads no temperature data**: the C1 panel is built without
LST columns, and LST enters only after C1 passes.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.analysis.spec import DATA, CL, fit, row
from src.analysis.corridors import cells_xs, build_panel, FE_C, CL_PORT, ZC, SAMPLE

## R1 · Do predicted routes run along real roads? (prespec §6a)

Cross-section of cells outside the exclusions. ◇ Go on to C1 if the on-route coefficient is
positive with t ≥ 3. `flat_cost` and the balance table are for information.

In [ ]:
r1, xs_paths = {}, {}
for v in ["main", "flat_cost"]:
    xs = cells_xs(v)
    xs = xs[~xs.near_mine & ~xs.near_port & xs.road_hp.notna() & (xs.GID_0 > 0)].copy()
    xs_paths[v] = p = f"{DATA}/_corridors_xs_{v}.parquet"
    xs.to_parquet(p, index=False)
    r = row(fit("log_road_hp ~ on_route | GID_0^mine_dist_band", se=CL_PORT, data=p, base=False), "on_route")
    r1[v] = dict(est=r["est"], se=r["se"], t=r["est"] / r["se"], on_route_cells=int(xs.on_route.sum()), cells=len(xs))
    if v == "main":
        xs_main = xs
r1 = pd.DataFrame(r1).T
print(f"◇ R1: t = {r1.loc['main', 't']:.1f} -> {'pass' if r1.loc['main', 't'] >= 3 and r1.loc['main', 'est'] > 0 else 'FAIL'}")
r1

In [ ]:
bal = {}
for c in ["slope", "land_share", "port_dist_km"]:
    rb = row(fit(f"{c} ~ on_route | GID_0^mine_dist_band", se=CL_PORT, data=xs_paths["main"], base=False), "on_route")
    bal[c] = dict(off_route_mean=xs_main.loc[xs_main.on_route == 0, c].mean(), difference=rb["est"], se=rb["se"])
pd.DataFrame(bal).T.round(3)

## Gate C1 · First stage of lights on recentered corridor exposure (prespec §7)

◇ Continue to LST only if F ≥ 70 with port-catchment clustering. ADM1-clustered F and the
coefficients by distance band are for information.

In [ ]:
P = build_panel("main")
assert not any(n.startswith(("lst", "ar_", "d_lst", "glass")) for n in pq.read_schema(P).names)

c1 = {}
for lab, se in [("port catchment (gate)", CL_PORT), ("ADM1", CL)]:
    r = row(fit(f"log_ntl ~ {ZC} | {FE_C}", se=se, data=P, subset=SAMPLE), ZC)
    c1[lab] = dict(est=r["est"], se=r["se"], F=(r["est"] / r["se"]) ** 2)
c1 = pd.DataFrame(c1).T
print(f"◇ C1: F = {c1.iloc[0]['F']:.1f} -> {'pass: go to LST' if c1.iloc[0]['F'] >= 70 else 'FAIL: stop Part 1'}")
c1

In [ ]:
bands = fit(f"log_ntl ~ z_20_100 + z_100_300 + z_300_plus | {FE_C}", se=CL_PORT, data=P, subset=SAMPLE)
bands.tidy().set_index("variable")[["estimate", "std_error", "p_value"]]